# Playground Series S6E8 (Predicting Smartphone Addiction) / 「[S6E8] Top 20 Formula: Dual Master Rank Blend」解説版

- **コンペ**: [Predicting Smartphone Addiction (Playground Series - Season 6 Episode 8)](https://www.kaggle.com/competitions/playground-series-s6e8)（2,914チーム・残り6日）
- **元notebook**: [[S6E8] Top 20 Formula: Dual Master Rank Blend](https://www.kaggle.com/code/souvikdbiswas/s6e8-top-20-formula-dual-master-rank-blend)
- **原著者**: SOUVIK_D_BISWAS
- **スコア**: Public 0.97128（Best 0.97128, V1）・実行時間28秒・CPUのみ
- **ライセンス**: Apache 2.0

## 手法の概要

**モデルを一切学習しません。** 既に公開されている2本の高スコア提出CSV（`blend1.csv` = 0.97127、`blend2.csv` = 0.97128）を
読み込み、両者を **Rank-01 パーセンタイル正規化** してから 0.45 : 0.55 の凸結合で混ぜ、
提出ファイルを書き出すだけの5セルのnotebookです。実行28秒。

短いですが、**「AUCというメトリックの性質を利用した最小限のアンサンブル」** の教材としては非常に分かりやすく、
同時に **「この手のブレンドが実際にはどれだけ効いているのか」を疑う練習台** にもなります（後述）。

## 評価指標

**タスク**: 表形式データからスマートフォン依存（`addicted_label`）を二値予測する。
**指標**: **ROC-AUC（ROC曲線下面積）**。

ROC-AUC は「無作為に選んだ陽性サンプル1つと陰性サンプル1つを比べたとき、
陽性の方に高いスコアを付けられる確率」に等しい指標です。ここから2つの重要な性質が出ます。

1. **順位（ランク）だけで決まる**: 予測値そのものではなく大小関係しか見ない。
   したがって単調増加な変換（定数倍、対数、パーセンタイル化）を施しても**AUCは1ミリも変わらない**。
2. **キャリブレーションを問わない**: 「確率0.9」が本当に9割当たるかは評価されない。

**なぜこの指標か**: 依存の有無のようなラベルは通常**クラス不均衡**で、accuracy だと
「全員を非依存と予測」でも高得点が出てしまいます。また、実運用では
「上位何％に介入するか」の閾値が後から決まるため、**閾値に依存しない**AUCが自然な選択になります。

**この手法が指標をどう最適化しているか**: AUCが順位だけで決まるなら、
異なるモデルの出力を混ぜるときに**生の確率値をそのまま平均するのは危険**です。
片方が0〜1に広く散らばり、もう片方が0.4〜0.6に固まっていると、平均は前者に支配されます。
Rank-01 変換 `R(p) = (rank(p) - 1) / (N - 1)` は両者を強制的に一様分布 [0,1] に揃えるため、
**指定した重み（0.45 : 0.55）が意図どおりの影響力になる**——これがこのnotebookの唯一かつ中心的なアイデアです。

> ⚠️ これは**学習目的の解説付き写し**です。コード本体は原著のまま変更しておらず、実行もしていません（出力セルは空）。
> 実行には原著者の添付データセット（`smartphone-addiction-best-ot-datas`）に含まれる blend1.csv / blend2.csv が必要です。


# Kaggle Playground Series S6E8: Master Rank Blend Ensemble [0.9713+]
### Rank-01 Percentile Normalization & Convex Weight Optimization

**Competition**: Kaggle Playground Series - Season 6 Episode 8 (Predicting Smartphone Addiction)  
**Evaluation Metric**: Area Under the ROC Curve (ROC-AUC)  

---

### Executive Summary & Ensemble Strategy

This notebook combines two top-performing predictions (`blend1` with Public LB score **0.97127** and `blend2` with Public LB score **0.97128**) into an optimized **Rank-01 Percentile Ensemble**.

| Model Stream | Public LB Score | Assigned Weight | Optimization Strategy |
| :--- | :--- | :--- | :--- |
| **Blend 1 Stream** | **0.97127** | `0.45` | Rank-01 Percentile Normalized |
| **Blend 2 Stream** | **0.97128** | `0.55` | Rank-01 Percentile Normalized |
| **Master Ensemble** | **0.9713+** | `1.00` | Rescaled Rank-01 Probability Vector |

---

### Table of Contents
1. Environment & Library Initialization  
2. Universal Data & Prediction Ingestion  
3. Statistical Correlation & Distribution Analysis  
4. Rank-01 Normalization & Convex Blending  
5. Final Submission Export & Verification  


## 1. Environment & Library Initialization

### 【解説】セル1: ライブラリの読み込みと出力設定

**何をしているか**: `numpy` / `pandas`（表データ操作）、`seaborn` / `matplotlib`（可視化）、
そして `scipy.stats` から `rankdata`（順位付け）・`pearsonr`（ピアソン相関）・`spearmanr`（スピアマン順位相関）を読み込みます。
最後に警告の抑制とUTF-8出力の設定を行っています。

**なぜそうするのか**: このnotebookで実質的に使うのは `rankdata` だけで、他は補助です。
`rankdata` は配列を「小さい順に1, 2, 3…」の順位に変換する関数で、同点は平均順位を割り当てます
（例: 値が [10, 20, 20, 30] なら順位は [1, 2.5, 2.5, 4]）。この同点処理が後の Rank-01 変換の基礎になります。

**初心者向け補足**: `warnings.filterwarnings('ignore')` は便利ですが**諸刃の剣**です。
「将来のバージョンで動かなくなる」といった重要な警告まで消えます。
公開notebookでは出力を綺麗に見せるために多用されますが、自分の実験では最後に一度外して確認する習慣が安全です。


In [ ]:
# Core libraries for data handling, statistical analysis, and plotting
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import rankdata, pearsonr, spearmanr

# Suppress non-critical warnings for clean output
warnings.filterwarnings('ignore')

# Ensure UTF-8 output encoding across environments
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Global visual style configuration
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("Libraries initialized successfully.")

## 2. Universal Data & Prediction Ingestion

### 【解説】セル2: 入力CSVの場所を自動探索して読み込む

**何をしているか**: `resolve_blend_paths()` という関数が、複数の候補ディレクトリ
（`/kaggle/input`、カレント、親ディレクトリなど）を `os.walk` で再帰的に走査し、
ファイル名が `blend1.csv` / `blend2.csv` に一致するものを探して最初に見つかったパスを返します。
見つからなければ `FileNotFoundError` を投げます。

**なぜそうするのか**: Kaggleでは添付データセットのマウントパスが
`/kaggle/input/<dataset-slug>/...` となり、フォークした人が別のデータセットを付け替えると変わります。
ハードコードすると他人の環境で即座に壊れるため、**ファイル名で探しに行く**方が頑健です。
見つからないときに黙って続行せず**例外で止める**のも良い設計（fail-fast）です。

**⚠️ 批判的に見るべき点**: `os.walk` は `/kaggle/input` 全体を走査するので、
別のデータセットに同名の `blend1.csv` があれば**意図しないファイルを掴む**可能性があります。
本来はファイルサイズや行数、あるいはハッシュ値で「掴んだのが本当に想定のファイルか」を確認すべきです。
また、この2つのCSVの中身が**何のモデルの出力なのか**はnotebookからは一切分かりません。
「0.97127 のブレンド」という情報しかなく、**再現性という意味では実質ブラックボックス**です。


In [ ]:
# Universal Path Resolver: Automatically searches Kaggle input datasets and local folders
def resolve_blend_paths():
    search_dirs = ["best_blende", ".", "..", "/kaggle/input", r"/kaggle/input/datasets/souvikdbiswas/smartphone-addiction-best-ot-datas"]
    b1_path, b2_path = None, None
    
    for d in search_dirs:
        if not os.path.exists(d): continue
        for root, dirs, files in os.walk(d):
            for f in files:
                f_lower = f.lower()
                if f_lower == 'blend1.csv' and not b1_path:
                    b1_path = os.path.join(root, f)
                elif f_lower == 'blend2.csv' and not b2_path:
                    b2_path = os.path.join(root, f)
                    
    if not b1_path or not b2_path:
        raise FileNotFoundError("Unable to locate blend1.csv or blend2.csv in environment directories.")
        
    return b1_path, b2_path

b1_file, b2_file = resolve_blend_paths()
print(f"Blend 1 File (0.97127) : {b1_file}")
print(f"Blend 2 File (0.97128) : {b2_file}")

df1 = pd.read_csv(b1_file)
df2 = pd.read_csv(b2_file)

print(f"Blend 1 Shape: {df1.shape}")
print(f"Blend 2 Shape: {df2.shape}")

## 3. Statistical Correlation & Distribution Analysis

### 【解説】セル3: 2つの予測の相関を測る

**何をしているか**: 2本の予測列 `addicted_label` について、
**ピアソン相関**（値そのものの直線的な関係の強さ）と
**スピアマン相関**（順位に変換してからのピアソン相関）を計算して表示します。

**なぜそうするのか**: アンサンブルが効く条件は「個々が十分強く、かつ**誤りが互いに独立**であること」です。
2つの予測が完全に一致（相関1.0）していれば、混ぜても何も変わりません。
相関を測るのは、**混ぜる価値があるかの事前チェック**です。

**AUCが指標なので、見るべきはスピアマンの方**です。ピアソンは値のスケールに引きずられますが、
AUCは順位しか見ないため、「順位がどれくらい違うか」を測るスピアマンが本質的です。

**⚠️ 批判的に見るべき点**: このセルは相関を**表示するだけ**で、その値に応じて処理を分岐させていません。
仮にスピアマン相関が 0.999 だったとしても、次のセルは何事もなかったように 0.45 : 0.55 で混ぜます。
本来なら「相関が高すぎるなら混ぜても無意味」という判断につなげるべきセルです。
実際、2本とも 0.97127 / 0.97128 という**ほぼ同一のスコア**で、
どちらも公開されたブレンドの派生である可能性が高く、相関は極めて高いと予想されます。


In [ ]:
# Compute Pearson and Spearman Rank Correlations
p_corr, _ = pearsonr(df1['addicted_label'], df2['addicted_label'])
s_corr, _ = spearmanr(df1['addicted_label'], df2['addicted_label'])

print("="*70)
print("STATISTICAL CORRELATION BETWEEN BLEND 1 & BLEND 2")
print("="*70)
print(f"  * Pearson Linear Correlation  : {p_corr:.6f}")
print(f"  * Spearman Rank Correlation   : {s_corr:.6f}")
print("="*70)

# Plot probability density distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.kdeplot(df1['addicted_label'], label='Blend 1 (0.97127)', ax=axes[0], color='#2ecc71', fill=True, alpha=0.3)
sns.kdeplot(df2['addicted_label'], label='Blend 2 (0.97128)', ax=axes[0], color='#9b59b6', fill=True, alpha=0.3)
axes[0].set_title("Probability Density Overlay", fontweight='bold')
axes[0].set_xlabel("Predicted Probability")
axes[0].legend()

diff = df2['addicted_label'] - df1['addicted_label']
sns.histplot(diff, bins=100, ax=axes[1], color='#3498db', kde=True)
axes[1].set_title("Prediction Difference Histogram (Blend 2 - Blend 1)", fontweight='bold')
axes[1].set_xlabel("Difference")

plt.tight_layout()
plt.show()

## 4. Rank-01 Normalization & Convex Blending

### Mathematical Formulation
Because the competition metric is **ROC-AUC**, predictions depend strictly on the relative rank ordering of test instances. We map raw model predictions $p_i$ to uniform percentile ranks $R(p_i)$:

$$R(p_i) = \frac{\text{Rank}(p_i) - 1}{N - 1} \in [0, 1]$$

We then apply a convex combination using optimal weights $w_1 = 0.45$ and $w_2 = 0.55$:

$$S_{\text{ensemble}} = w_1 \cdot R_1(p) + w_2 \cdot R_2(p) \quad \text{where } w_1 + w_2 = 1.0$$

### 【解説】セル4: Rank-01 パーセンタイル正規化と凸結合

**何をしているか**: 本notebookの中核です。

```python
def rank01(array):
    return (rankdata(array) - 1.0) / (len(array) - 1.0)
```

`rankdata` で 1〜N の順位に変換し、1を引いて 0〜N-1 にし、N-1 で割って **[0, 1] の一様分布**にします。
これを両方の予測に適用し、`0.45 * r1 + 0.55 * r2` で重み付き平均を取ります。

**なぜそうするのか（What/Whyの核心）**:
- **なぜランク化するのか**: 生の確率のまま平均すると、分布の広い方が結果を支配します。
  たとえばモデルAの出力が [0.01, 0.99] に散らばり、モデルBが [0.45, 0.55] に固まっていると、
  単純平均はほぼAの順位になり、Bの重みは名目上のものになってしまいます。
  ランク化すれば両者とも一様分布になるので、**指定した重みがそのまま影響力になります**。
- **なぜ凸結合（重みの和が1）か**: AUCはスケール不変なので、和が1である必要は実は**ありません**
  （全体を2倍しても順位は不変）。和を1に揃えるのは、結果を [0, 1] に収めて解釈しやすくするための慣習です。
- **なぜ 0.45 : 0.55 なのか**: コメントには「0.97128 の blend2 にわずかに大きい重み」とありますが、
  **この重みはnotebook内で最適化されていません**。ハードコードされた定数です。

**⚠️ 最も重要な批判点**: このnotebookの最終スコアは **0.97128** で、これは blend2 単独のスコアと**同一**です。
つまり「ブレンドによる改善は、公開LBの小数点第5位までの精度では観測されていない」ということです。
タイトルの "Top 20 Formula" が示唆するほどの効果は、**このスコア表示からは確認できません**。
相関の高い2本を混ぜても情報が増えないという、ensemble の基本原則どおりの結果と読むのが自然です。
学ぶべきは手法そのものよりも、**「スコアが変わっていないことに気づけるか」**という読み方の方です。


In [ ]:
# Rank-01 Percentile Normalization function
def rank01(array):
    return (rankdata(array) - 1.0) / (len(array) - 1.0)

# Apply Rank-01 transform to both input streams
r1 = rank01(df1['addicted_label'].values)
r2 = rank01(df2['addicted_label'].values)

# Optimal convex weights (giving slightly higher weight to Blend 2 at 0.97128)
w1 = 0.45
w2 = 0.55

raw_blend = w1 * r1 + w2 * r2
final_predictions = rank01(raw_blend)

print("Rank-01 Ensemble blending completed successfully.")

## 5. Final Submission Export & Verification

### 【解説】セル5: 提出ファイルの書き出しと検証

**何をしているか**: `id` と混合後の `addicted_label` からなる DataFrame を作り `submission.csv` に保存、
続いて行数と欠損値の数を表示します。

**なぜそうするのか**: 提出前の最小限のサニティチェックです。
行数が test の件数と合っているか、`NaN` が混入していないか——この2点は提出エラーの二大原因です。
`rankdata` は入力に `NaN` があると想定外の順位を付けるため、ここでの欠損チェックには実質的な意味があります。

**改善の余地**: 検証としては最低限です。加えるべきものとして、
(1) `id` が blend1 と blend2 で**同じ順序**か（このコードは `df1['id']` を無条件に使っており、
2つのCSVで行順が違えば予測とIDが**ずれたまま**提出されます）、
(2) 出力の分布が [0, 1] に収まっているか、
(3) 提出ファイルのハッシュ値を記録して過去の提出との重複を検出、などが挙げられます。
特に (1) は**このnotebookに実在するリスク**で、`pd.merge` で id を突き合わせるのが安全な書き方です。


In [ ]:
# Create Kaggle submission DataFrame
submission = pd.DataFrame({
    'id': df1['id'],
    'addicted_label': final_predictions
})

output_file = "submission.csv"
submission.to_csv(output_file, index=False)

print(f"Submission successfully written to '{output_file}'")
print(f"  * Total Instance Count : {len(submission):,}")
print(f"  * Missing Value Count  : {submission['addicted_label'].isna().sum()}")
print(f"  * Probability Range    : [{submission['addicted_label'].min():.6f}, {submission['addicted_label'].max():.6f}]")
print(f"  * Mean Expectation     : {submission['addicted_label'].mean():.6f}")

print("\nFirst 10 Rows of Master Blend Output:")
display(submission.head(10))